In [1]:
import re
from pathlib import Path

from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


C:\Users\ARTI\AppData\Local\Temp\ipykernel_3760\23720791.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
BASE_DIR = Path.cwd()
PDF_PATH = BASE_DIR / "aakash-resume.pdf"   # <- put any PDF name here
CHROMA_DIR = BASE_DIR / "chroma_db"
COLLECTION_NAME = "resume"

EMBED_MODEL = "nomic-embed-text"
OLLAMA_BASE_URL = "http://localhost:11434"

CHUNK_SIZE = 500
CHUNK_OVERLAP = 80

print("Working folder :", BASE_DIR)
print("PDF exists     :", PDF_PATH.exists())


Working folder : c:\Users\ARTI\Desktop\Abhishek\RAG-chatbot1
PDF exists     : True


In [3]:
loader = PyPDFLoader(str(PDF_PATH))
documents = loader.load()

print("Pages loaded:", len(documents))
print("Type of each item:", type(documents[0]))


Pages loaded: 1
Type of each item: <class 'langchain_core.documents.base.Document'>


In [4]:
doc = documents[0]
print("page_content (first 250 chars):\n", doc.page_content[:250])
print("\nmetadata:", doc.metadata)


page_content (first 250 chars):
 AAKASH MAHARANIDIN GUPTA 
Electronics & Telecommunication Engineering 
+91-7559409264   |   aakashguptamdeen41@gmail.com   |   MUMBAI 
LinkedIn   |   GitHub   |   LeetCode 
SUMMARY 
Electronics & Telecommunication Engineering graduate with hands-on e

metadata: {'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-09-23T13:46:51+00:00', 'author': 'Un-named', 'moddate': '2026-09-23T13:46:51+00:00', 'source': 'c:\\Users\\ARTI\\Desktop\\Abhishek\\RAG-chatbot1\\aakash-resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}


In [5]:
for d in documents:
    text = re.sub(r"[ \t]+", " ", d.page_content)   # many spaces/tabs -> one space
    text = re.sub(r"\n{3,}", "\n\n", text)          # 3+ blank lines -> one blank line
    d.page_content = text.strip()                    # trim the ends

print("Cleaned page 1 length:", len(documents[0].page_content), "characters")


Cleaned page 1 length: 2599 characters


In [6]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    separators=["\n\n", "\n", " ", ""],
    is_separator_regex=False,
)

chunks = splitter.split_documents(documents)

for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = i

print("Total chunks:", len(chunks))


Total chunks: 6


In [7]:
for chunk in chunks:
    print(f"chunk {chunk.metadata['chunk_id']} | page {chunk.metadata['page'] + 1} | {len(chunk.page_content)} chars")

print("\n=== Chunk 0 ===")
print(chunks[0].page_content)


chunk 0 | page 1 | 419 chars
chunk 1 | page 1 | 496 chars
chunk 2 | page 1 | 444 chars
chunk 3 | page 1 | 469 chars
chunk 4 | page 1 | 428 chars
chunk 5 | page 1 | 470 chars

=== Chunk 0 ===
AAKASH MAHARANIDIN GUPTA 
Electronics & Telecommunication Engineering 
+91-7559409264 | aakashguptamdeen41@gmail.com | MUMBAI 
LinkedIn | GitHub | LeetCode 
SUMMARY 
Electronics & Telecommunication Engineering graduate with hands-on expertise in Python, AI systems, and web development. 
Experienced in building RAG-based chatbots, desktop automation tools, and Flask applications. Proficient in LLM integration, vector


In [8]:
for n in range(len(chunks) - 1):
    a, b = chunks[n], chunks[n + 1]
    if a.metadata["page"] == b.metadata["page"]:
        print(f"\nEnd of chunk {n}  : ...{a.page_content[-80:]!r}")
        print(f"Start of chunk {n+1}: {b.page_content[:80]!r}...")
        break
    


End of chunk 0  : ...' automation tools, and Flask applications. Proficient in LLM integration, vector'
Start of chunk 1: 'databases, and document processing. Also experienced in SEO optimization from di'...


In [9]:
embeddings = OllamaEmbeddings(model=EMBED_MODEL, base_url=OLLAMA_BASE_URL)

chunk_texts = [c.page_content for c in chunks]
chunk_vectors = embeddings.embed_documents(chunk_texts)

print("Number of vectors   :", len(chunk_vectors))
print("Dimensions per vector:", len(chunk_vectors[0]))
print("First 5 numbers of chunk 0:", chunk_vectors[0][:5])


Number of vectors   : 6
Dimensions per vector: 768
First 5 numbers of chunk 0: [-0.062453806, 0.06025137, -0.15038356, -0.069238886, 0.06011156]


In [10]:
def make_store():
    return Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings,
        persist_directory=str(CHROMA_DIR),
        collection_metadata={"hnsw:space": "cosine"},
    )


# Remove any old collection so re-running never creates duplicates
make_store().delete_collection()

store = make_store()
ids = ["resume-chunk-" + str(c.metadata["chunk_id"]) for c in chunks]
store.add_documents(documents=chunks, ids=ids)

print("{} chunks stored in {}".format(store._collection.count(), CHROMA_DIR))


6 chunks stored in c:\Users\ARTI\Desktop\Abhishek\RAG-chatbot1\chroma_db


In [11]:
print("Collection metadata:", store._collection.metadata)

stored = store._collection.get(
    ids=["resume-chunk-0"],
    include=["embeddings", "documents", "metadatas"],
)
print("\nStored ID       :", stored["ids"][0])
print("Vector length   :", len(stored["embeddings"][0]))
print("Metadata        :", stored["metadatas"][0])
print("Text (first 100):", stored["documents"][0][:100])


Collection metadata: {'hnsw:space': 'cosine'}

Stored ID       : resume-chunk-0
Vector length   : 768
Metadata        : {'moddate': '2026-09-23T13:46:51+00:00', 'chunk_id': 0, 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-09-23T13:46:51+00:00', 'source': 'c:\\Users\\ARTI\\Desktop\\Abhishek\\RAG-chatbot1\\aakash-resume.pdf', 'total_pages': 1, 'author': 'Un-named', 'producer': 'www.ilovepdf.com', 'page_label': '1', 'page': 0}
Text (first 100): AAKASH MAHARANIDIN GUPTA 
Electronics & Telecommunication Engineering 
+91-7559409264 | aakashguptam
